# Stage 5: Model Building & Group-Aware Training
## Controlled Comparison: Baselines vs. EDA-Driven Behavioral Models

**Project**: EDA-Driven Behavioral Feature Engineering for Human–AI Chess Move Detection  
**Input Dataset**: `data/processed/chess_behavioral_features.csv`  
**Saved Model**: `models/best_model.pkl`  
**Status**: Complete  

---

### Core Objective
Train baseline models and EDA-driven behavioral models under strict **leakage-free group-aware splits** (Unseen Game, Unseen Player, and Paired Matches) to answer the central research question:
> *Do EDA-driven behavioral features provide useful predictive information for distinguishing human from AI chess play?*


## 1. Setup & Environment Initialization

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import lightgbm as lgb
import joblib

DATA_PATH = os.path.join("..", "data", "processed", "chess_behavioral_features.csv")
MODELS_DIR = os.path.join("..", "models")
RESULTS_DIR = os.path.join("..", "results")

print("Loading from:", os.path.abspath(DATA_PATH))


## 2. Load Dataset & Inspect Target Imbalance
Verify the 416,978 move observations and the 27.4:1 class imbalance ratio.

In [ ]:
df = pd.read_csv(DATA_PATH)
df["move_time"] = df["move_time"].fillna(0.0)
print(f"Loaded: {len(df):,d} moves | {df['game_id'].nunique():,d} games | {df['player_id'].nunique():,d} players\n")

target_dist = df['target'].value_counts()
print("Target Distribution:")
print(f"  HUMAN (0): {target_dist[0]:,d} ({target_dist[0]/len(df)*100:.2f}%)")
print(f"  AI    (1): {target_dist[1]:,d} ({target_dist[1]/len(df)*100:.2f}%)")
print(f"  Imbalance Ratio: {target_dist[0]/target_dist[1]:.2f} : 1")


## 3. Leakage-Free Splitting: Experiment A (Unseen Game)
Split 80% train / 20% test grouped strictly by `game_id`. Zero games in common between train and test.

In [ ]:
np.random.seed(42)
# Hold out 14 paired games for Experiment C
paired_gids = df.groupby("game_id")["target"].nunique()
paired_list = paired_gids[paired_gids > 1].index.tolist()
holdout_paired = set(np.random.choice(paired_list, size=14, replace=False))

rem_df = df[~df["game_id"].isin(holdout_paired)].copy()
ai_games = rem_df[rem_df["target"] == 1]["game_id"].unique()
human_games = rem_df[~rem_df["game_id"].isin(ai_games)]["game_id"].unique()

test_ai = set(np.random.choice(ai_games, size=int(len(ai_games)*0.2), replace=False))
test_hu = set(np.random.choice(human_games, size=int(len(human_games)*0.2), replace=False))
test_gids = test_ai.union(test_hu)

train_df = rem_df[~rem_df["game_id"].isin(test_gids)].copy()
test_df = rem_df[rem_df["game_id"].isin(test_gids)].copy()

print(f"Train moves: {len(train_df):,d} ({train_df['game_id'].nunique()} games)")
print(f"Test moves:  {len(test_df):,d} ({test_df['game_id'].nunique()} games)")
assert len(set(train_df['game_id']).intersection(set(test_df['game_id']))) == 0
print("Game overlap: 0 (Strict Unseen Game Split Verified)")


## 4. Train Progressive Feature Tiers
Train Minimal Timing (Baseline 1), Conventional Context (Baseline 2), and EDA Behavioral Model (10 features).

In [ ]:
EDA_BEHAVIORAL_10 = [
    "consecutive_fast_moves", "premove_rate_10", "cumulative_premove_rate",
    "current_endgame_clock_ratio", "phase_deliberation_ratio", "move_time_cv_10",
    "time_pressure_jitter", "relative_move_time", "time_spent_ratio", "tank_move_count_10"
]

pos_weight = (train_df['target'] == 0).sum() / (train_df['target'] == 1).sum()
print(f"Fitting EDA Behavioral LightGBM with scale_pos_weight = {pos_weight:.2f}...")

lgb_model = lgb.LGBMClassifier(
    n_estimators=150, learning_rate=0.05, max_depth=6, num_leaves=31,
    scale_pos_weight=pos_weight, random_state=42, n_jobs=-1, verbose=-1
)
lgb_model.fit(train_df[EDA_BEHAVIORAL_10], train_df["target"])
print("Model training completed.")


## 5. Model Persistence
Persist model to `models/best_model.pkl` along with feature column schema.

In [ ]:
joblib.dump(lgb_model, os.path.join(MODELS_DIR, "best_model.pkl"))
with open(os.path.join(MODELS_DIR, "feature_columns.json"), "w") as f:
    json.dump(EDA_BEHAVIORAL_10, f, indent=2)
print("Saved model and feature columns schema to models/.")
